# Synthetic image generation with Gemini

This notebook generates **synthetic images** for the **Urban Disaster Monitor** dataset using the **Gemini 2.5 Flash Image** model. The images simulate disaster scenarios (floods) with animals (cows, horses, cats, dogs) in a documentary/photojournalistic style, to enrich the dataset and reduce class imbalance.

---
**Project:** [Urban Disaster Monitor](https://github.com/MariaCarolinass/urban-disaster-monitor) · **Authors:** Carolina Soares, João Galdino

## Option 1: Manual use (web interface)

1. Open [Gemini 2.5 Flash Image](https://ai.google.com/research/gemini).
2. Enter the prompts in the image generation section.
3. Adjust resolution and style as needed.
4. Generate and download the images for inclusion in the dataset.

## Option 2: Batch generation via API

The cells below use the **Google Generative AI API** to generate multiple images in sequence. You need an API key (Google AI Studio or Google Cloud).

### Setup and dependencies

Install the libraries:

In [ ]:
!pip install google-generativeai pillow

Get an API key from [Google AI Studio](https://aistudio.google.com/apikey) or [Google Cloud Console](https://console.cloud.google.com/). Set the `GOOGLE_API_KEY` environment variable or use the code below with `getpass` to enter the key securely.

### Image generation

The code below loads the prompts, calls the Gemini API, and saves each image to the output folder. Adjust `OUTPUT_DIR` and the `prompts` list as needed.

In [ ]:
# API key: use environment variable or enter interactively
import os
api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    try:
        import getpass
        api_key = getpass.getpass("Paste your GOOGLE_API_KEY and press Enter: ")
    except Exception:
        api_key = "YOUR_KEY_HERE"  # replace with your key for local testing
print("API key configured." if api_key and api_key != "YOUR_KEY_HERE" else "Set GOOGLE_API_KEY or replace YOUR_KEY_HERE.")

In [ ]:
import os
import google.generativeai as genai
from PIL import Image
from io import BytesIO
import time

genai.configure(api_key=api_key)

OUTPUT_DIR = "generated_images"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Documentary/photojournalistic style prompts for flood scenarios (Urban Disaster Monitor)
prompts = [
    "Realistic low-resolution documentary photo of a cow isolated in floodwater reflecting sunset colors, authentic disaster photo",
    "Documentary style photo of a white horse trapped in a pasture with floodwater, overcast sky, realistic proportions, authentic disaster scene",
    "Low-resolution documentary photo of two wet cats stranded on a rooftop surrounded by floodwater, realistic proportions, visible faces, authentic urban flood event",
    "Documentary photo of a kitten on a balcony during flood, water rising near railing, cloudy weather, clear body visible, authentic disaster photo",
    "Photojournalism style disaster photo of a German shepherd swimming in a flooded street, cars partially submerged, realistic proportions, natural lighting",
    "Low quality documentary style photo of a stray dog on a small wooden raft during flood, rescue scene, realistic body intact",
    "Documentary flood photo of cows standing in a partially submerged field, water up to their legs, cloudy sky, authentic proportions",
    "Authentic documentary photo of a brown horse standing in a flooded street after heavy rain, realistic proportions, clear body visible, imperfect disaster photo",
]

# Image generation model name (see Google AI docs for current versions)
MODEL_NAME = "gemini-2.5-flash-image-preview"
MAX_RETRIES = 2
DELAY_BETWEEN_REQUESTS = 1.0  # seconds (avoid rate limiting)

def generate_and_save_image(prompt, index, total):
    print(f"[{index}/{total}] {prompt[:60]}...")
    for attempt in range(MAX_RETRIES):
        try:
            model = genai.GenerativeModel(model_name=MODEL_NAME)
            response = model.generate_content(prompt)
            image_data = None
            if response.candidates:
                for part in response.candidates[0].content.parts:
                    if part.inline_data:
                        image_data = part.inline_data.data
                        break
            if image_data:
                image = Image.open(BytesIO(image_data))
                filename = os.path.join(OUTPUT_DIR, f"image_{index:04d}.png")
                image.save(filename)
                print(f"  → Saved: {filename}")
                return True
            if response.candidates and response.candidates[0].content.parts:
                for part in response.candidates[0].content.parts:
                    if part.text:
                        print(f"  Model replied (text): {part.text[:80]}...")
        except Exception as e:
            print(f"  Tentativa {attempt + 1}/{MAX_RETRIES} falhou: {e}")
        time.sleep(DELAY_BETWEEN_REQUESTS)
    print(f"  FAILED after {MAX_RETRIES} attempts.")
    return False

for i, prompt in enumerate(prompts, start=1):
    generate_and_save_image(prompt, i, len(prompts))
    time.sleep(DELAY_BETWEEN_REQUESTS)

print("Generation complete.")
print(f"Images in: {os.path.abspath(OUTPUT_DIR)}")

### List generated images

Run the cell below to list files in the output folder.

In [ ]:
import glob

files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.png")))
print(f"Total: {len(files)} images")

for f in files[:15]:
    print(f"  {os.path.basename(f)}")
    
if len(files) > 15:
    print(f"  ... and {len(files) - 15} more")